In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

model_path = "/model"

In [3]:
# HYPER-PARAMETERS
BATCH_SIZE = 64
LEARNING_RATE = 0.001
EPOCHS = 50

In [4]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}")

transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

dataset = datasets.ImageFolder(root="data/train", transform=transform)
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

Using device: cpu


In [5]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Input: 3 channels (RGB), Output: 16 feature maps, Kernel size: 3x3
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Input: 16 channels, Output: 32 feature maps
        self.conv2 = nn.Conv2d(
            in_channels=16, out_channels=32, kernel_size=3, padding=1
        )

        # Fully Connected layers (Image is 8x8 after two poolings of a 32x32 image)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)  # 10 output classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))

        x = torch.flatten(x, 1)

        # Apply relu to the output of the connected layer1
        x = F.relu(self.fc1(x))

        # Apply relu to the output of the connected layer2
        x = F.relu(self.fc2(x))

        logits = self.fc3(x)
        return logits

In [6]:
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [7]:
def train_model():
    for epoch in range(EPOCHS):
        print(f"---------------Running epoch-----{epoch+1}")
        model.train()
        running_loss = 0.0
        for batch_idx, (images, labels) in enumerate(train_loader):
            # Move data to the configured device
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()  # Clear gradients from previous step
            loss.backward()
            optimizer.step()

train_model()

---------------Running epoch-----1
---------------Running epoch-----2
---------------Running epoch-----3
---------------Running epoch-----4
---------------Running epoch-----5
---------------Running epoch-----6
---------------Running epoch-----7
---------------Running epoch-----8
---------------Running epoch-----9
---------------Running epoch-----10
---------------Running epoch-----11
---------------Running epoch-----12
---------------Running epoch-----13
---------------Running epoch-----14
---------------Running epoch-----15
---------------Running epoch-----16
---------------Running epoch-----17
---------------Running epoch-----18
---------------Running epoch-----19
---------------Running epoch-----20
---------------Running epoch-----21
---------------Running epoch-----22
---------------Running epoch-----23
---------------Running epoch-----24
---------------Running epoch-----25
---------------Running epoch-----26
---------------Running epoch-----27
---------------Running epoch-----28
-

# Predict with the trained model

*  Use the image dataset to predict on both on validation set and training set

# Predict on the training set


In [ ]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

dataset = datasets.ImageFolder(root="data/train", transform=transform)
dataset_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.eval()
correct = 0
total = 0

In [ ]:
with torch.inference_mode():
    for images, labels in dataset_loader:
        # Move data to the configured location
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = 100 * correct / total
print(f"Training Accuracy:{accuracy:.2f}")

Validation Accuracy:96.16


# Predict on the validation set

In [10]:
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]
)

dataset = datasets.ImageFolder(root="data/test", transform=transform)
dataset_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.eval()
correct = 0
total = 0

In [11]:
with torch.inference_mode():
    for images, labels in dataset_loader:
        #Move data to the configured location
        images, labels = images.to(device),labels.to(device)
        
        outputs = model(images)
        predictions = torch.argmax(outputs,dim=1)
        
        total += labels.size(0)
        correct+=(predictions == labels).sum().item()
        
accuracy = 100 * correct / total
print(f"Validation Accuracy:{accuracy:.2f}")

Validation Accuracy:66.02


#### Without augmentation
* - 96% accuracy on training set and 66.02% accuracy on testing set,clearly the model is overfitting
* - Decision: Choose to augment the data, and train the model again 

# Data Augmentation

In [33]:
train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

dataset = datasets.ImageFolder(root="data/train", transform=train_transform)
dataset_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.train()
correct = 0
total = 0

In [31]:
def train_model_aug():
    for epoch in range(EPOCHS):
        print(f"---------------Running epoch-----{epoch+1}")
        model.train()
        running_loss = 0.0
        for batch_idx, (images, labels) in enumerate(dataset_loader):
            # Move data to the configured device
            images, labels = images.to(device), labels.to(device)

            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward pass and optimization
            optimizer.zero_grad()  # Clear gradients from previous step
            loss.backward()
            optimizer.step()

train_model_aug()

---------------Running epoch-----1
---------------Running epoch-----2
---------------Running epoch-----3
---------------Running epoch-----4
---------------Running epoch-----5
---------------Running epoch-----6
---------------Running epoch-----7
---------------Running epoch-----8
---------------Running epoch-----9
---------------Running epoch-----10
---------------Running epoch-----11
---------------Running epoch-----12
---------------Running epoch-----13
---------------Running epoch-----14
---------------Running epoch-----15
---------------Running epoch-----16
---------------Running epoch-----17
---------------Running epoch-----18
---------------Running epoch-----19
---------------Running epoch-----20
---------------Running epoch-----21
---------------Running epoch-----22
---------------Running epoch-----23
---------------Running epoch-----24
---------------Running epoch-----25
---------------Running epoch-----26
---------------Running epoch-----27
---------------Running epoch-----28
-

# Predict on the training set


In [45]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)
dataset = datasets.ImageFolder(root="data/train", transform=transform)
dataset_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.eval()
correct = 0
total = 0

In [46]:
with torch.inference_mode():
    for images, labels in dataset_loader:
        # Move data to the configured location
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = 100 * correct / total
print(f"Training accuracy:{accuracy:.2f}")

Training accuracy:83.46


# Predict on Validation Set

* - Prediction on the training set

In [47]:
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)
dataset = datasets.ImageFolder(root="data/test", transform = transform)
dataset_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

model.eval()
correct = 0
total = 0

In [48]:
with torch.inference_mode():
    for images, labels in dataset_loader:
        # Move data to the configured location
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        predictions = torch.argmax(outputs, dim=1)

        total += labels.size(0)
        correct += (predictions == labels).sum().item()

accuracy = 100 * correct / total
print(f"Training accuracy:{accuracy:.2f}")

Training accuracy:77.87


# Change the architecture of model to regularize the connected layers

In [ ]:
"""
How does augmentation strength affect the generalization
of a CNN trained on a small CIFAR-10 dataset?

4 Experiments
  No augmentation
  Moderate augmentation
  Strong augmentation
  Bad augmentation

Generalization gap = training accuracy - validation accuracy
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# HYPER-PARAMETERS
HYPER_PARAMETERS = {
    "batch_size": 64,
    "learning_rate": 0.001,
    "epochs": 30,
    "seed": 42,
}

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else ("mps" if torch.backends.mps.is_available() else "cpu")
)
print(f"Using device: {device}")


# ---------------------------------------------------------------- transforms

# Deterministic part. This is the model's input contract, so BOTH
# train and validation must have it.
BASE = [
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
]

# Random transforms operate on PIL images, so they go BEFORE ToTensor().
AUGMENTATIONS = {
    "no_aug": [],
    "moderate_aug": [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
    ],
    "strong_aug": [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4),
        transforms.RandomRotation(15),
    ],
    # Label-breaking on purpose: an upside-down car is not a photograph
    # of a car, so the label stops matching the pixels.
    "bad_aug": [
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(90),
    ],
}

def build_train_transform(name):
    return transforms.Compose(AUGMENTATIONS[name] + BASE)


# Validation is never augmented. We are measuring generalization to
# unseen images, not to distorted ones.
val_transform = transforms.Compose(BASE)


# -------------------------------------------------------------------- model


class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # Input: 3 channels (RGB), Output: 16 feature maps, Kernel size: 3x3
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Input: 16 channels, Output: 32 feature maps
        self.conv2 = nn.Conv2d(
            in_channels=16, out_channels=32, kernel_size=3, padding=1
        )

        # Input: 32 channels, Output: 64 feature maps
        self.conv3 = nn.Conv2d(
            in_channels=32, out_channels=64, kernel_size=3, padding=1
        )

        # Fully Connected layers (Image is 4x4 after three poolings of 32x32)
        self.fc1 = nn.Linear(64 * 4 * 4, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)  # 10 output classes

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # 32x32 -> 16x16
        x = self.pool(F.relu(self.conv2(x)))  # 16x16 -> 8x8
        x = self.pool(F.relu(self.conv3(x)))  # 8x8   -> 4x4

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))

        # Raw logits. CrossEntropyLoss applies softmax internally, so
        # applying it here as well would flatten the gradients.
        logits = self.fc3(x)
        return logits


# ----------------------------------------------------------------- accuracy


def evaluate(model, loader):
    """Accuracy over a loader. No gradients, no weight updates."""
    model.eval()
    correct = 0
    total = 0

    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            predictions = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    return 100 * correct / total


# ----------------------------------------------------------------- training


def run_experiment(name):
    """Train one model under one augmentation setting."""
    # Same seed for every experiment, so any difference in the gap comes
    # from the augmentation and not from initialisation luck.
    torch.manual_seed(HYPER_PARAMETERS["seed"])

    train_dataset = datasets.ImageFolder(
        root="data/train", transform=build_train_transform(name)
    )
    train_loader = DataLoader(
        train_dataset, batch_size=HYPER_PARAMETERS["batch_size"], shuffle=True
    )

    # A second view of the training data with NO augmentation. Training
    # accuracy has to be measured on clean images or it is not comparable
    # to validation accuracy.
    train_eval_dataset = datasets.ImageFolder(
        root="data/train", transform=val_transform
    )
    train_eval_loader = DataLoader(
        train_eval_dataset, batch_size=HYPER_PARAMETERS["batch_size"], shuffle=False
    )

    val_dataset = datasets.ImageFolder(root="data/test", transform=val_transform)
    val_loader = DataLoader(
        val_dataset, batch_size=HYPER_PARAMETERS["batch_size"], shuffle=False
    )

    model = SimpleCNN().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=HYPER_PARAMETERS["learning_rate"]
    )

    print(f"\n{'=' * 52}\n{name}\n{'=' * 52}")

    for epoch in range(HYPER_PARAMETERS["epochs"]):
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            # Gradients accumulate by default, so clear last batch's first.
            optimizer.zero_grad()

            logits = model(images)
            loss = loss_fn(logits, labels)

            # Walks the grad_fn chain and fills every .grad
            loss.backward()

            # Reads those .grad values and moves the weights
            optimizer.step()

            running_loss += loss.item()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            train_acc = evaluate(model, train_eval_loader)
            val_acc = evaluate(model, val_loader)
            avg_loss = running_loss / len(train_loader)
            print(
                f"epoch {epoch + 1:3d}  loss {avg_loss:.4f}  "
                f"train {train_acc:6.2f}  val {val_acc:6.2f}  "
                f"gap {train_acc - val_acc:6.2f}"
            )

    train_acc = evaluate(model, train_eval_loader)
    val_acc = evaluate(model, val_loader)

    return {
        "name": name,
        "train_acc": train_acc,
        "val_acc": val_acc,
        "gap": train_acc - val_acc,
    }


def main():
    results = []

    for name in AUGMENTATIONS:
        results.append(run_experiment(name))

    print(f"\n{'=' * 52}\nRESULTS\n{'=' * 52}")
    print(f"{'experiment':<16}{'train':>9}{'val':>9}{'gap':>9}")
    for r in results:
        print(
            f"{r['name']:<16}{r['train_acc']:>9.2f}"
            f"{r['val_acc']:>9.2f}{r['gap']:>9.2f}"
        )

if __name__ == "__main__":
    main()

Using device: cpu

no_aug
epoch   1  loss 1.5445  train  52.00  val  51.01  gap   0.99
epoch   5  loss 0.7906  train  74.53  val  69.29  gap   5.24
epoch  10  loss 0.4880  train  85.63  val  72.38  gap  13.25
epoch  15  loss 0.3050  train  90.24  val  71.20  gap  19.04
epoch  20  loss 0.1911  train  93.09  val  70.54  gap  22.55
epoch  25  loss 0.1225  train  96.22  val  71.50  gap  24.72
epoch  30  loss 0.1025  train  97.51  val  71.33  gap  26.18

moderate_aug
epoch   1  loss 1.6732  train  49.05  val  49.25  gap  -0.20
epoch   5  loss 1.0302  train  67.09  val  66.28  gap   0.81
epoch  10  loss 0.8187  train  73.53  val  71.96  gap   1.57
epoch  15  loss 0.7205  train  76.47  val  74.24  gap   2.23
epoch  20  loss 0.6585  train  80.32  val  77.20  gap   3.12
epoch  25  loss 0.6239  train  80.27  val  77.37  gap   2.90
epoch  30  loss 0.5927  train  81.74  val  77.82  gap   3.92

strong_aug
epoch   1  loss 1.7823  train  45.75  val  45.44  gap   0.31
epoch   5  loss 1.2130  train  62

* - Moderate augmentation provides the best generalization-performance tradeoff. Strong augmentation further reduces the train–validation gap, but slightly hurts validation accuracy.